<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 將 LIT 與 Gemma 一起使用

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/responsible/docs/alignment/lit"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on Generative AI</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/responsible/lit_gemma.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/responsible/lit_gemma.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemini%2Fgemma-cookbook%2Fmain%2Fresponsible%2Flit_gemma.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemini/gemma-cookbook/blob/main/responsible/lit_gemma.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td> <td>    <a target="_blank" href="https://codelabs.developers.google.com/codelabs/responsible-ai/lit-gemma"><img src="https://www.tensorflow.org/images/codelabs_logo.png" height="24" width="48" />Learn in Codelabs</a>
</td>
</table>

生成式人工智慧產品相對較新，其行為可能存在多種差異
早期形式的軟體。这使得探索机器学习变得很重要
正在使用的模型，檢查模型行為的範例並進行調查
驚喜。
學習可解釋性工具（LIT；[網站][lit-web]、[GitHub][lit-gh]）
是一個用於調試和分析 ML 模型以了解原因和方式的平台
他們的行為方式就是他們的行為方式。
在這裡，您將學習如何設定 LIT 以充分利用 Google
[Gemma模型][gemma]透過使用序列顯著性模組來分析不同的
prompt 工程方法。
[lit-web]：https://pair-code.github.io/lit
[lit-gh]：https://github.com/PAIR-code/lit
[寶石]：https://ai.google.dev/gemma
[keras-nlp]：https://keras.io/keras_nlp/

# 設定 LIT 以調試 Gemma 提示

*注意：您可能會看到一些表單的警告*

```
ERROR: pip's dependency resolver does not currently take into account all the 
packages that are installed. This behaviour is the source of the following 
dependency conflicts.
bigframes 0.21.0 requires scikit-learn>=1.2.2, but you have scikit-learn 1.0.2 
which is incompatible.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 8.14.0 
which is incompatible.
```

*這些可以安全地忽略。 *

## 安裝 LIT 和Keras NLP

這個notebook使用Gemma的KerasNLP 實現（更多關於如何
請在下面進行設定）。您將需要最新版本的 `keras` (3.0+)
`keras-nlp` (0.12+) 和`lit-nlp` (1.2+)，以及Kaggle 帳戶來下載
基礎模型。

In [ ]:
# Keras is included in Colab runtimes, but needs to be updated to to v3.0+.
# LIT and Keras NLP are not icnldued by default and must be installed.
# Running this cell may require you to restart your session to ensure the newer
# packages are imported correctly.
! pip install -q -U "keras >= 3.0, <4.0" "keras-nlp >= 0.14" "lit-nlp >= 1.2"

### Kaggle 訪問

KerasNLP 將其預訓練模型權重儲存在Kaggle 上。的
使用 [`kagglehub` 封裝](https://github.com/Kaggle/kagglehub#authenticate)
使用此服務進行身份驗證。請務必同時接受許可協議
從您的 Kaggle 帳戶取得 [Gemma](https://www.kaggle.com/models/keras/gemma)。
有關如何設定 Kaggle 的更多信息，請參閱最後的附錄。
帳戶。

In [ ]:
import kagglehub

kagglehub.login()

## 設定 LIT

LIT 提供了一個函數`make_notebook_widget()`來設定我們的prompt
notebook 上下文中的偵錯工具。
LIT 提供了 dataset 範例 prompt，並附有連結的教學
稍後將在本文檔中介紹。
請參閱下面的評論，以了解如何設定小部件以使用不同的模型和/或
datasets。

In [ ]:
from lit_nlp.examples.prompt_debugging import notebook as lit_pdbnb

# The following function initializes a LIT Notebook Widget. It's configured by
# two required positional arguments:
#
# * `datasets_config`: A list of strings containing the dataset names and
#       paths to load from, as "dataset:path", where path can be a URL or a
#       local file path. The example below uses a special value,
#       `sample_prompts`, to load the example prompts provided in the LIT
#       distribution; no other special values are supported.
# * `models_config`: A list of strings containing the model names and paths to
#       load from, as "model:path", where path can be a URL, a local file path,
#       or the name of a preset for the configured deep learning framework.
#
# LIT supports salience computation for KerasNLP and Hugging Face Transformers
# models running on TensorFlow or PyTorch. Note that all models passed to the
# `models_config` parameter will be loaded using the same framework and runtime.
# You can cofnigre these with the following keywork arguments.
#
# * `dl_framework`: Must be one of "kerasnlp" or "transformers".
# * `dl_runtime`: Must be one of "tensorflow" or "torch".
#
# Changing the `dl_framework` value will affect the authentication method used
# to access Gemma model weights.

lit_widget = lit_pdbnb.make_notebook_widget(
    ['sample_prompts'],
    ["gemma_2b_it:gemma_1.1_instruct_2b_en"],
    dl_framework="kerasnlp",
    dl_runtime="tensorflow",
    batch_size=1,
    max_examples=5,
    precision="bfloat16",
)

現在您可以在 Colab cell 中渲染 UI。

In [ ]:
lit_widget.render()

<IPython.core.display.Javascript object>

# 透過序列顯著性進行提示調試

文字轉文字大語言模型 (LLM)，例如 Gemma，採用輸入序列
以 [tokenized][tokenization] 文本的形式產生新的tokens
邏輯後續或完成。
[顯著性方法][顯著性-可探索]可讓您檢查物件的哪些部分
輸入對於模型產生的輸出的不同部分都很重要。 LIT 的[Sequence Salience module][lit-seq-sal]擴展了這些方法來解釋
序列在多個粒徑層級的重要性：從 tokens 到
單字到句子及其他。
您可以使用上面 cell 中的 LIT 來嘗試序列顯著性
自己的模組。要獲得更有指導的學習體驗，您可以長期關注
使用[_使用序列顯著性進行提示偵錯_教學][seq-sal-tutorial]
就在這個Colab。
有關序列顯著性如何工作的更多學術和技術信息，
查看[我們的論文][seq-sal-paper]。

[lit-seq-sal]：https://pair-code.github.io/lit/documentation/components.html#sequence-salience
[顯著性-可探索]：https://pair.withgoogle.com/explorables/saliency/
[seq-sal-paper]：https://arxiv.org/abs/2404.07498
[seq-sal-教學]：https://pair-code.github.io/lit/tutorials/sequence-salience/
[token化]：https://arxiv.org/pdf/1808.06226.pdf

# Appendix: Accessing Gemma on Kaggle Hub

此notebook 使用本文檔中Gemma 的KerasNLP 實作。 KerasNLP 將其預先訓練的模型權重儲存在 Kaggle 上，Gemma 需要
存取這些權重的身份驗證和許可證確認。
以下說明將引導您了解如何設定 Kaggle 帳戶並
使用`kagglehub` 套件對Kaggle 進行身份驗證。
1.  如果您沒有帳戶，請建立 Kaggle 帳戶
* 轉至：https://www.kaggle.com/account/login?phase=startRegisterTab * 使用您喜歡的任何註冊方法來設定您的帳戶。1. 請求訪問Gemma
* 確保您已使用上述帳號登入Kaggle * 進入同意頁面：https://www.kaggle.com/models/google/gemma/license/consent * 選擇「透過Kaggle帳戶驗證」選項（預設選擇），然後按一下下一步 * 填寫同意書（名字和姓氏欄位位於頂部） * 使用底部的複選框確認政策 * 點選底部的「接受」按鈕即可獲得存取權限 * 這應該會將您重定向到模型頁面 (https://www.kaggle.com/models/google/gemma)1. 建立 API token
* 確保您已使用上面建立的帳戶登入Kaggle * 進入設定頁面：https://www.kaggle.com/settings *向下捲動至API部分 * 使用「建立新 token」按鈕觸發token生成 * 使用螢幕選單將服務產生的名為 kaggle.json 的 JSON 檔案儲存到您的電腦上 * JSON 檔案是具有兩個屬性（使用者名稱和金鑰）的對象，稍後您將需要這兩個屬性來對其服務進行身份驗證1. 使用您的 API token 憑證在 Colab 中向 kagglehub 進行身份驗證
* 轉至 LIT 序列銷售Colab：https://colab.sandbox.google.com/github/google/generative-ai-docs/blob/main/site/en/gemma/docs/lit_gemma.ipynb#scrollTo=yKw8gDsh_nVR * 連接到 GPU runtime * 對於 Gemma 2B，您可以使用免費套餐 T4 runtime * 對於 Gemma 7B，您需要預付 Colab 計算積分或 Colab Pro 帳戶才能使用 V100、L4 或 A100 GPU * 執行`kagglehub`代碼cell顯示一個 HTML 表單，要求您輸入使用者名稱和token * 從上一個步驟下載的`kaggle.json` 檔案複製`username` 字段，並將其貼上到表單中的`username` 字段中 * 從上一個步驟下載的`kaggle.json` 檔案複製`key` 字段，並將其貼上到表單中的`token` 字段中 * 點選登入按鈕將這些憑證儲存在您的runtime中
每當 Colab runtime 中斷連線時，您都需要重複最後一步，因為斷開連線會清除儲存憑證的快取。